# Topic Modeling with NMF

## Setup and Imports

In [22]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

In [23]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [24]:
model_type = 'nmf' # or 'nmf'
data_home = "../input"


In [25]:
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [26]:
OHCO = ['doc_title', 'para_num', 'sentence_num', 'token_num']
SENTS = OHCO[:3]
PARAS = OHCO[:2]
STORIES = OHCO[:1]

BAG = PARAS

In [27]:
BAG

['doc_title', 'para_num']

In [28]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str  \
doc_title para_num sentence_num token_num                                   
ASHPUTTEL 0        0            0            ('The', 'DT')   DT       The   
                                1           ('wife', 'NN')   NN      wife   
                                2             ('of', 'IN')   IN        of   
                                3              ('a', 'DT')   DT         a   
                                4           ('rich', 'JJ')   JJ      rich   
...                                                    ...  ...       ...   
TOM THUMB 21       3            40            ('s', 'VBZ')  VBZ         s   
                                41            ('no', 'DT')   DT        no   
                                42         ('place', 'NN')   NN     place   
                                43          ('like', 'IN')   IN      like   
                                44          ('HOME', 'NN')   NN      HOME   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ  
...                                            ...       ...  
TOM THUMB 21       3            40               s        VB  
                                41              no        DT  
                                42           place        NN  
                                43            like        IN  
                                44            home        NN  

[101046 rows x 5 columns]

In [29]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title para_num                                                   
ASHPUTTEL 0         wife man end drew daughter girl watch afterwar...
          1         work daylight water fire sisters sorts ways ev...
          2         father fair wife s daughters clothes diamonds ...
          3         king land feast days son bride sisters hair sh...
          4                             peas ashes maiden door garden
...                                                               ...
TOM THUMB 17        wolf night house drain kitchen pantry ate dran...
          18        shout noise wolf everybody house clatter man mind
          19        woodman wife noise crack door wolf woodman axe...
          20                                             riches world
          21        son plenty clothes ones journey home father mo...

[889 rows x 1 columns]

## Create Vector Space

In [30]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]

['mine',
 'have',
 'towards',
 'side',
 'made',
 'whither',
 'detail',
 'am',
 'has',
 'inc']

In [31]:
# count_engine = CountVectorizer(max_df=.9, min_df=2, stop_words=my_stop_words) # Got some advice from clause to lower min ax max df because corpus ins amll
# count_model = count_engine.fit_transform(DOCS.doc_str)
# TERMS = count_engine.get_feature_names_out()
# VOCAB = pd.DataFrame(index=TERMS)
# VOCAB.index.name = 'term_str'
# DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
# DTM

In [32]:
# Used claude code to help with tfidf engine and model because I want nmf to get better topics than lda
tfidf_engine = TfidfVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words)
tfidf_model = tfidf_engine.fit_transform(DOCS.doc_str)
TERMS = tfidf_engine.get_feature_names_out()
TFIDF = tfidf_engine.fit_transform(DOCS.doc_str)
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM = pd.DataFrame(tfidf_model.toarray(), index=DOCS.index, columns=TERMS)
DTM


account  advice      air  alas  ale  anger  animals  \
doc_title para_num                                                        
ASHPUTTEL 0             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          1             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          2             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          3             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          4             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
...                     ...     ...      ...   ...  ...    ...      ...   
TOM THUMB 17            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          18            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          19            0.0     0.0  0.12911   0.0  0.0    0.0      0.0   
          20            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          21            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   

                    answer  apple  apples  ...  woods  word  words      work  \
doc_title para_num                         ...                                 
ASHPUTTEL 0            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          1            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.280609   
          2            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          3            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          4            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
...                    ...    ...     ...  ...    ...   ...    ...       ...   
TOM THUMB 17           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          18           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          19           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          20           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          21           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   

                       world  wretch   ye  year  years  youth  
doc_title para_num                                             
ASHPUTTEL 0         0.000000     0.0  0.0   0.0    0.0    0.0  
          1         0.000000     0.0  0.0   0.0    0.0    0.0  
          2         0.000000     0.0  0.0   0.0    0.0    0.0  
          3         0.000000     0.0  0.0   0.0    0.0    0.0  
          4         0.000000     0.0  0.0   0.0    0.0    0.0  
...                      ...     ...  ...   ...    ...    ...  
TOM THUMB 17        0.000000     0.0  0.0   0.0    0.0    0.0  
          18        0.000000     0.0  0.0   0.0    0.0    0.0  
          19        0.105347     0.0  0.0   0.0    0.0    0.0  
          20        1.000000     0.0  0.0   0.0    0.0    0.0  
          21        0.000000     0.0  0.0   0.0    0.0    0.0  

[889 rows x 573 columns]

## Generate Model with 20 Topics

In [33]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [34]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(tfidf_model)

## THETA

In [35]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

doc_title,SNOW-WHITE AND ROSE-RED,THE TRAVELLING MUSICIANS,SNOW-WHITE AND ROSE-RED,BRIAR ROSE,THE GOLDEN GOOSE,ASHPUTTEL,THE PINK,THE FOUR CLEVER BROTHERS,BRIAR ROSE,THE ADVENTURES OF CHANTICLEER AND PARTLET
para_num,6,3,5,5,3,13,2,10,6,10
topic_id,,,,,,,,,,
T0,0.000000,0.021958,0.000000,0.075257,0.057036,0.042829,0.128672,0.037133,0.113231,0.000000
T1,0.000000,0.000000,0.007075,0.000000,0.009713,0.140137,0.000880,0.002426,0.006145,0.000000
T2,0.000000,0.000058,0.000000,0.002794,0.009472,0.024685,0.069716,0.012078,0.012446,0.000000
T3,0.245193,0.051408,0.101665,0.000000,0.021218,0.015852,0.015964,0.006065,0.000000,0.140844
T4,0.123198,0.000000,0.288381,0.000000,0.054583,0.006762,0.091085,0.000000,0.000000,0.000000


## PHI

In [36]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

term_str,girls,storm,table,sides,straw,parson,judge,brother,frock,dwarfs
topic_id,,,,,,,,,,
T0,0.076253,0.009458,0.121034,0.024323,0.023277,0.029699,0.033959,0.121575,0.021163,0.039574
T1,0.011192,0.000000,0.029192,0.001730,0.000000,0.000000,0.000000,0.000000,0.005255,0.000000
T2,0.003397,0.000000,0.001301,0.000000,0.000000,0.000000,0.000000,0.131254,0.035698,0.000000
T3,0.011620,0.003057,0.042061,0.004580,0.043879,0.019856,0.000000,0.000000,0.008542,0.049263
T4,0.007989,0.006091,0.000000,0.000000,0.000000,0.000000,0.000000,0.006254,0.000000,0.000000


## Get Top Terms By Topic

In [ ]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS


## Save Files to Output

In [38]:
THETA.to_csv(f"{output_dir}/pg2591-THETA.csv", index=True)
PHI.to_csv(f"{output_dir}/pg2591-PHI.csv", index=True)
TOPICS.to_csv(f"{output_dir}/pg2591-TOPICS.csv", index=True)